# CLARIFYING MODELS DIRECTORY

In [1]:
from pathlib import Path
import shutil

LEGACY_N = Path(
    "/kaggle/working/runs/detect/runs/cardboard_compare/"
    "yolov8n/weights/best.pt"
)

MODEL_DIR = Path("/kaggle/working/amm_yolo/models")
MODEL_DIR.mkdir(parents=True, exist_ok=True)

if LEGACY_N.exists():
    destination = MODEL_DIR / "yolov8n_best.pt"
    shutil.copy2(LEGACY_N, destination)

    print("Existing YOLOv8n checkpoint rescued:", destination)
    print("Size (MB):", round(destination.stat().st_size / 1024**2, 2))
else:
    print("Legacy checkpoint not found at the expected path.")

    matches = [
        p for p in Path("/kaggle/working").rglob("best.pt")
        if "yolov8n" in p.parts and "weights" in p.parts
    ]

    for path in matches:
        print("Candidate:", path)

Legacy checkpoint not found at the expected path.


In [2]:
%pip install -q "ultralytics==8.4.184" roboflow pyyaml "pillow<13" "gym==0.25.2"

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gymnax 0.0.8 requires gym>=0.26, but you have gym 0.25.2 which is incompatible.
Note: you may need to restart the kernel to use updated packages.


# IMPORTING DEPENDENCIES

In [3]:
import sys
import gc
import json
import shutil
import zipfile
import statistics
import importlib
import importlib.metadata as metadata

from pathlib import Path

import numpy as np
import pandas as pd
import yaml
import torch

# Compatibility workaround for checkpoint loading in some environments.
# Explicitly import the PyTorch internal module before loading .pt files.
import torch._utils

import ultralytics
from ultralytics import YOLO
from roboflow import Roboflow

from IPython.display import FileLink, display


# Stable project directories for this notebook.
WORK_DIR = Path("/kaggle/working/amm_yolo")
RUNS_DIR = WORK_DIR / "runs"
MODEL_DIR = WORK_DIR / "models"
DATASET_DIR = WORK_DIR / "dataset_cardboard_v2"

for directory in [WORK_DIR, RUNS_DIR, MODEL_DIR]:
    directory.mkdir(parents=True, exist_ok=True)


MODEL_CHECKPOINTS = {
    "yolov8n": "yolov8n.pt",
    "yolov8s": "yolov8s.pt",
    "yolov8m": "yolov8m.pt",
}

print("Python executable:", sys.executable)
print("Python version:", sys.version.split()[0])
print("PyTorch version:", torch.__version__)
print("PyTorch location:", torch.__file__)
print("Ultralytics version:", ultralytics.__version__)
print("Ultralytics location:", ultralytics.__file__)
print("torch._utils available:", hasattr(torch, "_utils"))
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

print("Working directory:", WORK_DIR)
print("Models directory:", MODEL_DIR)

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
Python executable: /usr/bin/python3
Python version: 3.13.15
PyTorch version: 2.11.0+cu128
PyTorch location: /usr/local/lib/python3.13/dist-packages/torch/__init__.py
Ultralytics version: 8.4.184
Ultralytics location: /usr/local/lib/python3.13/dist-packages/ultralytics/__init__.py
torch._utils available: True
CUDA available: True
GPU: Tesla T4
Working directory: /kaggle/working/amm_yolo
Models directory: /kaggle/working/amm_yolo/models


# MODELS CHECK POINTS

In [4]:
try:
    smoke_test_model = YOLO("yolov8n.pt")
    print("Pretrained checkpoint loaded successfully.")

    del smoke_test_model
    gc.collect()

except Exception as exc:
    if "_utils" in str(exc):
        raise RuntimeError(
            "PyTorch checkpoint loading still fails. "
            "Do not start training yet. See the troubleshooting "
            "steps below for environment diagnostics."
        ) from exc
    raise

Pretrained checkpoint loaded successfully.


In [5]:
%pip check

bigframes 2.48.0 requires google-cloud-bigquery-storage, which is not installed.
gymnax 0.0.8 has requirement gym>=0.26, but you have gym 0.25.2.
google-colab 1.0.0 has requirement jupyter-server==2.20.0, but you have jupyter-server 2.12.5.
google-colab 1.0.0 has requirement pandas==2.2.3, but you have pandas 2.3.3.
google-colab 1.0.0 has requirement requests==2.32.4, but you have requests 2.34.2.
moviepy 1.0.3 has requirement decorator<5.0,>=4.0.2, but you have decorator 5.3.1.
Note: you may need to restart the kernel to use updated packages.


# EXTRACTING ARTIFACT ARCHIVES FOR ALL MODLES

In [6]:
# Extract a previously exported artifact archive, if attached as Kaggle input.
IMPORT_DIR = WORK_DIR / "imported_artifacts"
IMPORT_DIR.mkdir(parents=True, exist_ok=True)

for archive in Path("/kaggle/input").rglob("amm_yolo_artifacts.zip"):
    print("Found saved artifact archive:", archive)

    with zipfile.ZipFile(archive, "r") as zip_ref:
        zip_ref.extractall(IMPORT_DIR)

# Restore standardized model files from attached Kaggle inputs.
for model_name in MODEL_CHECKPOINTS:
    destination = MODEL_DIR / f"{model_name}_best.pt"

    if destination.exists():
        print(f"Already available: {destination.name}")
        continue

    direct_matches = list(
        Path("/kaggle/input").rglob(f"{model_name}_best.pt")
    )

    imported_matches = list(
        IMPORT_DIR.rglob(f"{model_name}_best.pt")
    )

    matches = direct_matches + imported_matches

    if matches:
        shutil.copy2(matches[0], destination)
        print("Restored:", model_name, "from", matches[0])

# Recover the previously trained YOLOv8n from the old run directory,
# if the current session still contains it.
legacy_candidates = [
    Path("/kaggle/working/runs/detect/runs/cardboard_compare/"
         "yolov8n/weights/best.pt"),
    Path("/kaggle/working/runs/cardboard_compare/"
         "yolov8n/weights/best.pt"),
]

for model_name in MODEL_CHECKPOINTS:
    destination = MODEL_DIR / f"{model_name}_best.pt"

    if destination.exists():
        continue

    candidates = [
        path for path in legacy_candidates
        if path.exists()
        and "yolov8n" in str(path)
    ] if model_name == "yolov8n" else []

    if not candidates:
        candidates = [
            p for p in Path("/kaggle/working").rglob("best.pt")
            if "weights" in p.parts and model_name in p.parts
        ]

    if candidates:
        source = max(candidates, key=lambda p: p.stat().st_mtime)
        shutil.copy2(source, destination)
        print(f"Recovered {model_name} from:", source)

print("\nAvailable model artifacts:")
for model_name in MODEL_CHECKPOINTS:
    path = MODEL_DIR / f"{model_name}_best.pt"
    print(model_name, "->", "Available" if path.exists() else "Not saved yet")


Available model artifacts:
yolov8n -> Not saved yet
yolov8s -> Not saved yet
yolov8m -> Not saved yet


# API KEY

In [7]:
from kaggle_secrets import UserSecretsClient

DATA_YAML = DATASET_DIR / "data.yaml"

if DATA_YAML.exists():
    print("Using the dataset already downloaded in this session.")

else:
    try:
        user_secrets = UserSecretsClient()
        api_key = user_secrets.get_secret("ROBOFLOW_API_KEY")

        if not api_key:
            raise ValueError("The Roboflow API key is empty.")

    except Exception as exc:
        raise RuntimeError(
            "Roboflow access is not configured. "
            "Check Kaggle Secrets: ROBOFLOW_API_KEY, "
            "and grant the Notebook access to it."
        ) from exc

    rf = Roboflow(api_key=api_key)

    project = (
        rf.workspace("dataset-t7hz7")
        .project("cardboard-eupc8")
    )

    dataset = project.version(2).download(
        "yolov8",
        location=str(DATASET_DIR),
    )

    DATA_YAML = Path(dataset.location) / "data.yaml"

assert DATA_YAML.exists(), f"Dataset YAML not found: {DATA_YAML}"

print("Dataset configuration:", DATA_YAML)

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to /kaggle/working/amm_yolo/dataset_cardboard_v2 in yolov8:: 100%|██████████| 1582/1582 [00:00<00:00, 11906.66it/s]

Dataset configuration: /kaggle/working/amm_yolo/dataset_cardboard_v2/data.yaml


# CHECKING FOR PATHS ERRORS  

In [8]:
config_path = Path(DATA_YAML)
config = yaml.safe_load(config_path.read_text())

print("Dataset configuration:")
print(config)

print("\nClass names:", config.get("names"))
print("Number of classes:", config.get("nc"))
print("Training split:", config.get("train"))
print("Validation split:", config.get("val"))
print("Test split:", config.get("test"))

assert "train" in config, "The YAML file has no training split."
assert "val" in config, "The YAML file has no validation split."
assert "names" in config, "The YAML file has no class names."

names = config["names"]
num_classes = len(names)

print("\nConfirmed number of classes:", num_classes)

if num_classes == 1:
    print("Single-class detection dataset confirmed.")

Dataset configuration:
{'names': ['cardboard'], 'nc': 1, 'roboflow': {'license': 'CC BY 4.0', 'project': 'cardboard-eupc8', 'url': 'https://universe.roboflow.com/dataset-t7hz7/cardboard-eupc8/dataset/2', 'version': 2, 'workspace': 'dataset-t7hz7'}, 'test': '../test/images', 'train': '../train/images', 'val': '../valid/images'}

Class names: ['cardboard']
Number of classes: 1
Training split: ../train/images
Validation split: ../valid/images
Test split: ../test/images

Confirmed number of classes: 1
Single-class detection dataset confirmed.


# CHECKPOINTS UPDATE

In [9]:

from pathlib import Path
from IPython.display import display, FileLink
from ultralytics import YOLO
import shutil
import json
import zipfile
import gc
import torch

# --------------------------------------------------
# 1. Paths and training configuration
# --------------------------------------------------
WORK_DIR = Path(WORK_DIR)
RUNS_DIR = Path(RUNS_DIR)
MODEL_DIR = Path(MODEL_DIR)
DATA_YAML = Path(DATA_YAML)

WORK_DIR.mkdir(parents=True, exist_ok=True)
RUNS_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

MODEL_CHECKPOINTS = {
    "yolov8n": "yolov8n.pt",
    "yolov8s": "yolov8s.pt",
    "yolov8m": "yolov8m.pt",
}

SAVE_PERIOD = 10
LATEST_BACKUP = WORK_DIR / "amm_yolo_recovery_latest.zip"


# --------------------------------------------------
# 2. Create ZIP backups without including old ZIPs
# --------------------------------------------------
def make_backup_zip(zip_path, current_run_dir=None):
    zip_path = Path(zip_path)
    temp_path = zip_path.with_name(zip_path.stem + "_building.zip")

    run_dirs = [RUNS_DIR / name for name in MODEL_CHECKPOINTS]
    if current_run_dir is not None:
        run_dirs.append(Path(current_run_dir))

    # Remove duplicate directories while preserving order.
    run_dirs = list(dict.fromkeys(run_dirs))

    with zipfile.ZipFile(
        temp_path, "w", compression=zipfile.ZIP_DEFLATED
    ) as zf:

        # Include all saved model weights and JSON metadata.
        for file_path in MODEL_DIR.iterdir():
            if (
                file_path.is_file()
                and file_path.suffix.lower() in {".pt", ".json"}
            ):
                zf.write(
                    file_path,
                    arcname=f"models/{file_path.name}"
                )

        # Include useful training results and configuration.
        useful_files = [
            "results.csv",
            "args.yaml",
            "results.png",
            "confusion_matrix.png",
            "PR_curve.png",
            "P_curve.png",
            "R_curve.png",
            "F1_curve.png",
        ]

        for run_dir in run_dirs:
            if not run_dir.exists():
                continue

            for filename in useful_files:
                file_path = run_dir / filename
                if file_path.is_file():
                    zf.write(
                        file_path,
                        arcname=f"runs/{run_dir.name}/{filename}"
                    )

    # Replace the previous ZIP only after the new one is complete.
    temp_path.replace(zip_path)


# --------------------------------------------------
# 3. Save recovery checkpoints during training
# --------------------------------------------------
def make_epoch_backup_callback(model_name):

    def backup_after_epoch(trainer):
        try:
            epoch_number = int(trainer.epoch) + 1

            # Create a recovery package every 10 epochs.
            # Also create one at the final epoch.
            if (
                epoch_number % SAVE_PERIOD != 0
                and epoch_number != int(trainer.epochs)
            ):
                return

            last_path = Path(trainer.last)
            best_path = Path(trainer.best)

            saved_last = MODEL_DIR / f"{model_name}_last.pt"
            saved_best = MODEL_DIR / f"{model_name}_best.pt"

            if last_path.is_file():
                shutil.copy2(last_path, saved_last)

            if best_path.is_file():
                shutil.copy2(best_path, saved_best)

            progress_info = {
                "model": model_name,
                "status": "in_progress",
                "epochs_completed": epoch_number,
                "epochs_requested": int(trainer.epochs),
                "last_checkpoint": str(saved_last),
                "best_weights": str(saved_best),
                "run_directory": str(trainer.save_dir),
            }

            info_path = MODEL_DIR / f"{model_name}_training_info.json"
            with open(info_path, "w", encoding="utf-8") as file:
                json.dump(progress_info, file, indent=2)

            current_run_dir = Path(trainer.save_dir)
            make_backup_zip(
                LATEST_BACKUP,
                current_run_dir=current_run_dir
            )

            print(
                f"\nRecovery backup updated: "
                f"{model_name}, epoch {epoch_number}"
            )
            print("Download the recovery ZIP if needed:")
            display(FileLink(str(LATEST_BACKUP)))

        except Exception as error:
            # A backup failure should not unnecessarily crash training.
            print(f"WARNING: Recovery backup failed: {error}")

    return backup_after_epoch


# --------------------------------------------------
# 4. Train, resume, or skip each model
# --------------------------------------------------
for model_name, checkpoint in MODEL_CHECKPOINTS.items():

    final_best = MODEL_DIR / f"{model_name}_best.pt"
    final_last = MODEL_DIR / f"{model_name}_last.pt"
    info_path = MODEL_DIR / f"{model_name}_training_info.json"

    run_dir = RUNS_DIR / model_name
    run_last = run_dir / "weights" / "last.pt"
    run_best = run_dir / "weights" / "best.pt"

    # Read training metadata, if available.
    info = {}
    if info_path.is_file():
        try:
            with open(info_path, "r", encoding="utf-8") as file:
                info = json.load(file)
        except (json.JSONDecodeError, OSError):
            info = {}

    # Skip a completed model, including one completed by the
    # previous version of the training cell.
    previously_completed = (
        final_best.is_file()
        and info_path.is_file()
        and info.get("status") in (None, "completed")
    )

    if previously_completed:
        if not final_last.is_file() and run_last.is_file():
            shutil.copy2(run_last, final_last)

        print(f"\nSkipping completed model: {model_name}")
        continue

    # Prefer the original run's last.pt. If the run directory is
    # unavailable, use the recovery copy if one was saved.
    recovery_last = (
        run_last if run_last.is_file()
        else final_last if final_last.is_file()
        else None
    )

    print(f"\n{'=' * 55}")
    print(f"Model: {model_name}")

    if recovery_last is not None:
        print(f"Resuming from checkpoint: {recovery_last}")
        model = YOLO(str(recovery_last))
    else:
        print("Starting a new training run.")
        model = YOLO(checkpoint)

    model.add_callback(
        "on_model_save",
        make_epoch_backup_callback(model_name)
    )

    try:
        if recovery_last is not None:
            # Resume the interrupted training, restoring its
            # saved training state instead of starting over.
            model.train(
                resume=True,
                data=str(DATA_YAML),
                save_period=SAVE_PERIOD,
            )
        else:
            model.train(
                data=str(DATA_YAML),
                epochs=100,
                imgsz=640,
                batch=8,
                patience=40,
                device=0,
                workers=2,
                seed=42,
                pretrained=True,
                project=str(RUNS_DIR),
                name=model_name,
                exist_ok=True,
                plots=True,
                save=True,
                save_period=SAVE_PERIOD,
            )

        # --------------------------------------------------
        # 5. Copy final weights AFTER this model finishes
        # --------------------------------------------------
        trainer = model.trainer
        actual_run_dir = Path(trainer.save_dir)

        best_source = Path(trainer.best)
        last_source = Path(trainer.last)

        if not best_source.is_file():
            best_source = actual_run_dir / "weights" / "best.pt"

        if not last_source.is_file():
            last_source = actual_run_dir / "weights" / "last.pt"

        if not best_source.is_file():
            raise FileNotFoundError(
                f"Final best.pt not found for {model_name}: "
                f"{best_source}"
            )

        if not last_source.is_file():
            raise FileNotFoundError(
                f"Final last.pt not found for {model_name}: "
                f"{last_source}"
            )

        shutil.copy2(best_source, final_best)
        shutil.copy2(last_source, final_last)

        # Mark the model completed only after both copies exist.
        completion_info = {
            "model": model_name,
            "status": "completed",
            "dataset_yaml": str(DATA_YAML),
            "epochs_requested": 100,
            "imgsz": 640,
            "batch": 8,
            "patience": 40,
            "best_weights": str(final_best),
            "last_checkpoint": str(final_last),
            "run_directory": str(actual_run_dir),
        }

        with open(info_path, "w", encoding="utf-8") as file:
            json.dump(completion_info, file, indent=2)

        # Create a separate backup after each model completes.
        completed_zip = (
            WORK_DIR / f"amm_yolo_backup_after_{model_name}.zip"
        )

        make_backup_zip(
            completed_zip,
            current_run_dir=actual_run_dir
        )

        print(f"\nCompleted: {model_name}")
        print(f"Best weights: {final_best}")
        print(f"Resume checkpoint: {final_last}")
        print("Download this completed-model backup:")
        display(FileLink(str(completed_zip)))

    except KeyboardInterrupt:
        print(f"\nTraining interrupted for {model_name}.")
        print("Look for the latest recovery ZIP below.")

        if LATEST_BACKUP.is_file():
            display(FileLink(str(LATEST_BACKUP)))

        raise

    finally:
        del model
        gc.collect()

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

print("\nTraining stage finished.")


Model: yolov8n
Starting a new training run.
New https://pypi.org/project/ultralytics/8.4.194 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.184 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/amm_yolo/dataset_cardboard_v2/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_

`get_trial_id` is meant to only be called inside a function that is executed by a Tuner or Trainer. Returning `None`.


      2/100      1.32G      1.357      1.356      1.543         59        640: 100% ━━━━━━━━━━━━ 69/69 14.1it/s 4.9s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 10/10 11.7it/s 0.9s
                   all        156        418      0.678      0.646        0.7      0.391

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      3/100      1.32G      1.386      1.389      1.615         25        640: 100% ━━━━━━━━━━━━ 69/69 14.3it/s 4.8s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 10/10 11.6it/s 0.9s
                   all        156        418      0.626      0.531      0.594      0.304

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      4/100      1.32G      1.366      1.344      1.591         36        640: 100% ━━━━━━━━━━━━ 69/69 13.9it/s 5.0s
                 Class     Images  Instances      Box(P  

/kaggle/working/amm_yolo/amm_yolo_recovery_latest.zip


      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     11/100      1.32G      1.298      1.172      1.553         33        640: 100% ━━━━━━━━━━━━ 69/69 13.6it/s 5.1s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 10/10 11.3it/s 0.9s
                   all        156        418      0.815      0.699      0.808      0.483

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     12/100      1.32G      1.272      1.141      1.512         15        640: 100% ━━━━━━━━━━━━ 69/69 13.7it/s 5.1s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 10/10 10.5it/s 0.9s
                   all        156        418      0.804      0.647      0.746      0.449

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     13/100      1.32G      1.296      1.163      1.542         34        640: 100% ━━━━━━━━━━━

...

Closing dataloader mosaic
albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01, method='weighted_average', num_output_channels=3), CLAHE(p=0.01, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8))

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     91/100      4.09G     0.7683     0.4821      1.246         10        640: 100% ━━━━━━━━━━━━ 69/69 4.8it/s 14.3s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 10/10 5.6it/s 1.8s
                   all        156        418      0.953      0.822      0.923      0.638

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     92/100      4.09G     0.7539     0.4559      1.225         16        640: 100% ━━━━━━━━━━━━ 69/69 5.0it/s 13.8s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 10/10 5.5it/s 1.8s
                   all   

/kaggle/working/amm_yolo/amm_yolo_recovery_latest.zip


100 epochs completed in 0.539 hours.
Optimizer stripped from /kaggle/working/amm_yolo/runs/yolov8m/weights/last.pt, 52.0MB
Optimizer stripped from /kaggle/working/amm_yolo/runs/yolov8m/weights/best.pt, 52.0MB

Validating /kaggle/working/amm_yolo/runs/yolov8m/weights/best.pt...
Ultralytics 8.4.184 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
Model summary (fused): 92 layers, 25,840,339 parameters, 0 gradients, 78.7 GFLOPs
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 10/10 3.0it/s 3.3s
                   all        156        418      0.904      0.876      0.936      0.651
Speed: 1.5ms preprocess, 10.8ms inference, 0.0ms loss, 4.1ms postprocess per image
Results saved to /kaggle/working/amm_yolo/runs/yolov8m

Completed: yolov8m
Best weights: /kaggle/working/amm_yolo/models/yolov8m_best.pt
Resume checkpoint: /kaggle/working/amm_yolo/models/yolov8m_last.pt
Download this completed-model backup:


/kaggle/working/amm_yolo/amm_yolo_backup_after_yolov8m.zip


Training stage finished.


# Training Evaluation

In [10]:
training_rows = []
EVAL_DEVICE = 0 if torch.cuda.is_available() else "cpu"
EVAL_IMGSZ = 640  # Match the training configuration in this original code.
EVAL_BATCH = 8    # Match the training configuration in this original code.

for model_name in MODEL_CHECKPOINTS:
    weights = MODEL_DIR / f"{model_name}_best.pt"

    if not weights.is_file():
        raise FileNotFoundError(
            f"Required checkpoint for {model_name} was not found: {weights}. "
            "Run/fix CHECKPOINTS UPDATE before evaluation."
        )

    print(f"Training evaluation: {model_name}")
    model = YOLO(str(weights))

    try:
        metrics = model.val(
            data=str(DATA_YAML),
            split="train",
            imgsz=EVAL_IMGSZ,
            batch=EVAL_BATCH,
            device=EVAL_DEVICE,
            workers=2,
            plots=True,
            project=str(RUNS_DIR / "training_evaluation"),
            name=model_name,
            exist_ok=True,
            verbose=True,
        )

        training_rows.append({
            "Model": model_name,
            "Precision": float(metrics.box.mp),
            "Recall": float(metrics.box.mr),
            "mAP50": float(metrics.box.map50),
            "mAP50-95": float(metrics.box.map),
            "Inference ms/image": float(metrics.speed["inference"]),
        })

    finally:
        del model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

training_evaluation = pd.DataFrame(training_rows)
if training_evaluation.empty:
    raise RuntimeError("Training evaluation produced no results.")

training_evaluation = training_evaluation.round(4).sort_values(
    "mAP50-95", ascending=False
)
display(training_evaluation)
training_evaluation.to_csv(
    WORK_DIR / "training_evaluation.csv", index=False
)
print("Training evaluation saved:", WORK_DIR / "training_evaluation.csv")

Training evaluation: yolov8n
Ultralytics 8.4.184 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
Model summary (fused): 73 layers, 3,132,707 parameters, 0 gradients, 8.6 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 698.9±133.3 MB/s, size: 18.7 KB)
val: Scanning /kaggle/working/amm_yolo/dataset_cardboard_v2/train/labels.cache... 550 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 550/550 192.2Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 69/69 12.7it/s 5.4s
                   all        550       1505      0.978      0.971      0.993      0.795
Speed: 1.1ms preprocess, 4.2ms inference, 0.0ms loss, 0.9ms postprocess per image
Results saved to /kaggle/working/amm_yolo/runs/training_evaluation/yolov8n
Training evaluation: yolov8s
Ultralytics 8.4.184 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
Model summary (fused): 72 layers, 11,125,971 parameters, 0 gradients, 28.4 

,Model,Precision,Recall,mAP50,mAP50-95,Inference ms/image
1,yolov8s,0.9814,0.9854,0.9937,0.8266,9.5856
2,yolov8m,0.9732,0.9667,0.9919,0.7958,25.5467
0,yolov8n,0.9784,0.9714,0.9927,0.7947,4.1732


Training evaluation saved: /kaggle/working/amm_yolo/training_evaluation.csv


# MODEL COMPARESION (Validation Evaluation)

In [11]:
comparison_rows = []

for model_name in MODEL_CHECKPOINTS:
    weights = MODEL_DIR / f"{model_name}_best.pt"

    if not weights.exists():
        print(f"Skipping {model_name}: no saved weights.")
        continue

    print(f"Evaluating {model_name}...")

    model = YOLO(str(weights))

    try:
        metrics = model.val(
            data=str(DATA_YAML),
            split="val",
            imgsz=640,
            batch=8,
            device=0,
            plots=True,
            project=str(RUNS_DIR / "validation"),
            name=model_name,
            exist_ok=True,
        )

        comparison_rows.append({
            "Model": model_name,
            "Precision": metrics.box.mp,
            "Recall": metrics.box.mr,
            "mAP50": metrics.box.map50,
            "mAP50-95": metrics.box.map,
            "Inference ms/image": metrics.speed["inference"],
        })

    finally:
        del model
        gc.collect()

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

comparison = pd.DataFrame(comparison_rows)

if not comparison.empty:
    comparison = comparison.round(4)
    display(comparison.sort_values("mAP50-95", ascending=False))

    comparison.to_csv(
        WORK_DIR / "model_comparison.csv",
        index=False,
    )
else:
    print("No saved model checkpoints were available to evaluate.")

Evaluating yolov8n...
Ultralytics 8.4.184 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
Model summary (fused): 73 layers, 3,132,707 parameters, 0 gradients, 8.6 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 670.3±240.6 MB/s, size: 16.9 KB)
val: Scanning /kaggle/working/amm_yolo/dataset_cardboard_v2/valid/labels.cache... 156 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 156/156 50.3Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 20/20 8.5it/s 2.4s
                   all        156        418      0.892      0.871      0.922      0.633
Speed: 1.7ms preprocess, 5.7ms inference, 0.0ms loss, 1.8ms postprocess per image
Results saved to /kaggle/working/amm_yolo/runs/validation/yolov8n
Evaluating yolov8s...
Ultralytics 8.4.184 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
Model summary (fused): 72 layers, 11,125,971 parameters, 0 gradients, 28.4 GFLOPs
val: Fast image ac

,Model,Precision,Recall,mAP50,mAP50-95,Inference ms/image
2,yolov8m,0.9061,0.8769,0.9334,0.6506,26.5717
1,yolov8s,0.8940,0.8469,0.9096,0.6401,10.6686
0,yolov8n,0.8922,0.8708,0.9217,0.6328,5.6922


# SAVING TRAINED MODELS RESULTS

In [12]:
ARTIFACT_ZIP = Path("/kaggle/working/amm_yolo_artifacts.zip")

with zipfile.ZipFile(
    ARTIFACT_ZIP,
    mode="w",
    compression=zipfile.ZIP_DEFLATED,
) as zip_file:

    # Include available final weights and training metadata.
    for path in MODEL_DIR.iterdir():
        if path.is_file():
            zip_file.write(path, arcname=path.name)

    comparison_csv = WORK_DIR / "model_comparison.csv"

    if comparison_csv.exists():
        zip_file.write(comparison_csv, arcname="model_comparison.csv")

print("Artifact archive created:", ARTIFACT_ZIP)
print("Archive size (MB):", round(ARTIFACT_ZIP.stat().st_size / 1024**2, 2))

display(FileLink(str(ARTIFACT_ZIP)))

Artifact archive created: /kaggle/working/amm_yolo_artifacts.zip
Archive size (MB): 142.43


/kaggle/working/amm_yolo_artifacts.zip

# Testing Evaluation

In [13]:
dataset_config_for_test = yaml.safe_load(Path(DATA_YAML).read_text(encoding="utf-8"))
if not dataset_config_for_test.get("test"):
    raise ValueError(
        "The dataset data.yaml does not define a 'test' split. "
        "A held-out test evaluation cannot run until that split is configured."
    )

testing_rows = []

for model_name in MODEL_CHECKPOINTS:
    weights = MODEL_DIR / f"{model_name}_best.pt"

    if not weights.is_file():
        raise FileNotFoundError(
            f"Required checkpoint for {model_name} was not found: {weights}. "
            "Run/fix CHECKPOINTS UPDATE before testing."
        )

    print(f"HELD-OUT TEST evaluation: {model_name}")
    model = YOLO(str(weights))

    try:
        metrics = model.val(
            data=str(DATA_YAML),
            split="test",
            imgsz=EVAL_IMGSZ,
            batch=EVAL_BATCH,
            device=EVAL_DEVICE,
            workers=2,
            plots=True,
            project=str(RUNS_DIR / "testing_evaluation"),
            name=model_name,
            exist_ok=True,
            verbose=True,
        )

        testing_rows.append({
            "Model": model_name,
            "Precision": float(metrics.box.mp),
            "Recall": float(metrics.box.mr),
            "mAP50": float(metrics.box.map50),
            "mAP50-95": float(metrics.box.map),
            "Inference ms/image": float(metrics.speed["inference"]),
        })

    finally:
        del model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

testing_evaluation = pd.DataFrame(testing_rows)
if testing_evaluation.empty:
    raise RuntimeError("Held-out testing evaluation produced no results.")

testing_evaluation = testing_evaluation.round(4).sort_values(
    "mAP50-95", ascending=False
)
display(testing_evaluation)
testing_evaluation.to_csv(
    WORK_DIR / "testing_evaluation.csv", index=False
)
print("Held-out testing evaluation saved:", WORK_DIR / "testing_evaluation.csv")

# Combine the three same-format tables into one report-friendly CSV while preserving
# the separate CSV files for easier analysis.
combined_evaluation = pd.concat(
    [
        training_evaluation.assign(Split="train"),
        comparison.assign(Split="val"),
        testing_evaluation.assign(Split="test"),
    ],
    ignore_index=True,
)
combined_evaluation = combined_evaluation[
    ["Split", "Model", "Precision", "Recall", "mAP50", "mAP50-95", "Inference ms/image"]
]
combined_evaluation.to_csv(
    WORK_DIR / "train_val_test_evaluation.csv", index=False
)
print("Combined train/val/test results saved:", WORK_DIR / "train_val_test_evaluation.csv")

HELD-OUT TEST evaluation: yolov8n
Ultralytics 8.4.184 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
Model summary (fused): 73 layers, 3,132,707 parameters, 0 gradients, 8.6 GFLOPs
WARNING ⚠️ val: Slow image access detected (ping: 0.0±0.0 ms, read: 12.0±9.6 MB/s, size: 17.1 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips
val: Scanning /kaggle/working/amm_yolo/dataset_cardboard_v2/test/labels... 79 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 79/79 1.0Kit/s 0.1s
val: New cache created: /kaggle/working/amm_yolo/dataset_cardboard_v2/test/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 10/10 5.9it/s 1.7s
                   all         79        185      0.906      0.838      0.924      0.653
Speed: 2.4ms preprocess, 6.0ms inference, 0.0ms loss, 4.4ms postprocess per image
Results saved to /kaggle/workin

,Model,Precision,Recall,mAP50,mAP50-95,Inference ms/image
2,yolov8m,0.9563,0.8272,0.9455,0.6568,27.4309
0,yolov8n,0.9065,0.8378,0.9238,0.6533,5.9782
1,yolov8s,0.9364,0.8649,0.9272,0.6521,12.4778


Held-out testing evaluation saved: /kaggle/working/amm_yolo/testing_evaluation.csv
Combined train/val/test results saved: /kaggle/working/amm_yolo/train_val_test_evaluation.csv


In [14]:
#!ls runs/detect